In [0]:
# Databricks notebook source
# MAGIC %md
# MAGIC # Data Setup - AI/BI Portfolio Assistant
# MAGIC Downloads raw data from GitHub (dbdemos-dataset) and creates tables in UC.
# MAGIC 
# MAGIC This notebook is idempotent: re-running it will skip downloads if data already exists.

In [0]:
%run ../../config

In [0]:
# Catalog is provided at deploy time; only create it if missing (avoids needing CREATE CATALOG on existing catalogs)
if catalog not in [r["catalog"] for r in spark.sql("SHOW CATALOGS").collect()]:
    spark.sql(f"CREATE CATALOG IF NOT EXISTS `{catalog}`")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS `{catalog}`.`{schema}`")
spark.sql(f"CREATE VOLUME IF NOT EXISTS `{catalog}`.`{schema}`.`{volume_name}`")
spark.sql(f"USE CATALOG `{catalog}`")
spark.sql(f"USE SCHEMA `{schema}`")

volume_path = f"/Volumes/{catalog}/{schema}/{volume_name}"

In [0]:
import requests
from pathlib import Path

GITHUB_REPO = "databricks-demos/dbdemos-dataset"
GITHUB_PATH = "aibi/dbdemos_aibi_fsi_portfolio_assistant_v2"

def download_folder_from_git(local_path: str, folder: str):
    """Download all parquet files from a GitHub folder to a local Volume path."""
    api_url = f"https://api.github.com/repos/{GITHUB_REPO}/contents/{GITHUB_PATH}/{folder}"
    resp = requests.get(api_url)
    resp.raise_for_status()
    
    files = [f for f in resp.json() if f['name'].endswith('.parquet')]
    dest = Path(local_path) / folder
    dest.mkdir(parents=True, exist_ok=True)
    
    for f in files:
        target = dest / f['name']
        if not target.exists():
            r = requests.get(f['download_url'])
            r.raise_for_status()
            target.write_bytes(r.content)
    
    print(f"  {folder}: {len(files)} parquet files ready")
    return str(dest)

In [0]:
# Map GitHub folder -> raw table name
data_sources = {
    "portfolios": "raw_portfolio",
    "securities": "raw_fundamentals",
    "prices": "raw_prices",
    "news": "raw_news",
    "news_ticker": "raw_news_ticker",
}

# Download only if volume is empty
import os as _os
needs_download = not _os.path.exists(f"{volume_path}/portfolios") or len(_os.listdir(f"{volume_path}/portfolios")) == 0

if needs_download:
    print(f"Downloading data from GitHub to {volume_path}...")
    for folder in data_sources:
        download_folder_from_git(volume_path, folder)
    print("Download complete!")
else:
    print("Data already exists in volume, skipping download.")

In [0]:
# Create raw_* tables from the downloaded parquet as MANAGED UC tables.
# (Reading via spark + saveAsTable avoids the external-LOCATION "cloud file
#  system scheme" error when pointing an external table at a /Volumes path.)
for folder, table_name in data_sources.items():
    path = f"{volume_path}/{folder}"
    df = spark.read.parquet(path)
    df.write.mode("overwrite").saveAsTable(table_name)
    count = spark.table(table_name).count()
    print(f"  {table_name}: {count} rows")


In [0]:
# portfolio: CamelCase -> snake_case, drop _rescued_data
spark.sql("""
    CREATE OR REPLACE TABLE portfolio AS
    SELECT
        Ticker AS ticker,
        CompanyName AS company_name,
        CompanyDescription AS company_description,
        CompanyWebsite AS company_website,
        CompanyLogo AS company_logo,
        Industry AS industry
    FROM raw_portfolio
""")

# fundamentals
spark.sql("""
    CREATE OR REPLACE TABLE fundamentals AS
    SELECT
        Ticker AS ticker,
        MarketCapitalization AS market_capitalization,
        OutstandingShares AS outstanding_shares
    FROM raw_fundamentals
""")

# prices
spark.sql("""
    CREATE OR REPLACE TABLE prices AS
    SELECT
        Ticker AS ticker,
        Date AS date,
        Open AS open,
        High AS high,
        Low AS low,
        Close AS close,
        AdjustedClose AS adjusted_close,
        `Return` AS `return`,
        Volume AS volume,
        SplitFactor AS split_factor
    FROM raw_prices
""")

# news
spark.sql("""
    CREATE OR REPLACE TABLE news AS
    SELECT
        ArticleId AS article_id,
        PublishedTime AS published_time,
        Source AS source,
        SourceUrl AS source_url,
        Title AS title,
        Sentiment AS sentiment,
        MarketSentiment AS market_sentiment
    FROM raw_news
""")

# news_ticker: explode ArticleIds array
spark.sql("""
    CREATE OR REPLACE TABLE news_ticker AS
    SELECT
        Ticker AS ticker,
        explode(ArticleIds) AS article_id
    FROM raw_news_ticker
""")

print("All tables created successfully!")
for t in ['portfolio', 'fundamentals', 'prices', 'news', 'news_ticker']:
    print(f"  {t}: {spark.table(t).count()} rows")